[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michael-palomino-tm/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/Mistral/RA2/IL2.2/1-memory-agent.ipynb)


# 1. Agentes con Memoria Conversacional en LangChain

## Objetivos de Aprendizaje
- Comprender por qué la memoria es crucial para crear agentes conversacionales efectivos.
- Aprender a gestionar el historial de una conversación (`chat_history`) con el `AgentExecutor`.
- Implementar un agente que recuerde interacciones pasadas para responder preguntas de seguimiento.
- Entender cómo LangChain pasa el contexto de la conversación al LLM.

## ¿Qué es la Memoria y Por Qué es Importante?

Por defecto, los LLMs y los agentes que hemos construido hasta ahora **no tienen estado (stateless)**. Cada vez que los invocamos, procesan la solicitud como si fuera la primera vez que interactúan con nosotros. No tienen recuerdo de preguntas o respuestas anteriores.

Esto es una gran limitación para crear asistentes o chatbots útiles. Un usuario espera poder hacer preguntas de seguimiento, referirse a información mencionada previamente y tener una conversación fluida. 

La **memoria** es el mecanismo que permite a un agente recordar interacciones pasadas. LangChain facilita enormemente la gestión de esta memoria. La forma más común de memoria es el **historial de chat (chat history)**, donde simplemente guardamos la lista de todos los mensajes de la conversación.

En este notebook, veremos cómo añadir esta capacidad a nuestro agente de LangChain.

### 1. Instalación y Configuración

In [1]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain langchain-classic langchain-openai python-dotenv wikipedia


In [2]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.mistral.ai/v1")
    os.environ.setdefault("LLM_MODEL", "mistral-small-latest")
    os.environ.setdefault("LLM_MODEL_SMALL", "ministral-8b-latest")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "mistral-small-latest"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. Herramientas y Agente (Sin Cambios)

La definición de las herramientas y la creación del agente son exactamente las mismas que en el notebook anterior. La magia de la memoria no está en la definición del agente, sino en **cómo lo ejecutamos**.

In [3]:
# LangChain v1: langchain.agents.create_agent (o langgraph)
from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Este prompt antes se descargaba con `hub.pull("hwchase17/openai-tools-agent")`.
# LangChain ahora bloquea por defecto la descarga de prompts públicos, porque
# pueden contener objetos serializados no confiables. Definirlo aquí evita esa
# dependencia de red y, sobre todo, deja a la vista qué le estamos diciendo al
# agente: instrucción de sistema, historial, entrada del usuario y el
# "scratchpad" donde el agente va anotando sus llamadas a herramientas.
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente útil."),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")

✅ Agente y herramientas listos.


### 3. Gestionando la Memoria Conversacional

Para que el agente recuerde, necesitamos hacer dos cosas:

1.  **Mantener un historial**: Crearemos una lista llamada `chat_history` para almacenar los mensajes.
2.  **Pasar el historial en cada llamada**: El `AgentExecutor` acepta un parámetro `chat_history`. LangChain se encarga de formatear esta lista y añadirla al prompt que se envía al LLM.

El formato del historial es una lista de objetos `BaseMessage` de LangChain. Los más comunes son `HumanMessage` (para el usuario) y `AIMessage` (para la respuesta del agente).

In [4]:
from langchain_core.messages import HumanMessage, AIMessage

# Iniciamos el historial de chat como una lista vacía
chat_history = []

#### Primera Interacción: Sin Historial

In [5]:
query1 = "Háblame del planeta Saturno"

response1 = agent_executor.invoke({
    "input": query1,
    "chat_history": chat_history
})

print(f"Respuesta 1: {response1['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'Saturno planeta'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Saturno'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)


Invoking: `get_wikipedia_summary` with `{'query': 'Saturn planeta'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Parece que hay un problema técnico con la herramienta para obtener información de Wikipedia. Sin embargo, puedo darte un resumen sobre **Saturno** basado en mi conocimiento general:

**Saturno** es el sexto planeta del Sistema Solar y el segundo más grande, conocido por sus espectaculares anillos compuestos principalmente de partículas de hielo y roca. Es un planeta gaseoso, compuesto principalmente de hidrógeno y helio, y destaca por su baja densidad, siendo el único planeta menos denso que el agua.

> Finished chain.
Respuesta 1: Parece que hay un problema técnico con la herramienta para obtener información de Wikipedia. Sin embargo, puedo darte un resumen sobre **Saturno** basado en mi conocimiento general:

**Saturno** es el sexto planeta del Sistema Solar y el segundo más grande, conocido por sus espectaculares anillos compuestos principalmente de partículas de hielo y roca. Es un planeta gaseoso, compuesto principalmente de hidrógeno y helio, y destaca por su baja densidad, siend

Ahora, actualizamos manualmente nuestro historial con la pregunta del usuario y la respuesta del agente.

In [6]:
chat_history.append(HumanMessage(content=query1))
chat_history.append(AIMessage(content=response1["output"]))

print("Historial actualizado.")

Historial actualizado.


#### Segunda Interacción: Con Historial

Ahora hacemos una pregunta de seguimiento. Fíjate que no mencionamos "Saturno", simplemente preguntamos "¿de qué están hechos sus anillos?"

In [7]:
query2 = "¿Y de qué están hechos sus anillos?"

response2 = agent_executor.invoke({
    "input": query2,
    "chat_history": chat_history
})

print(f"Respuesta 2: {response2['output']}")



> Entering new AgentExecutor chain...



Invoking: `get_wikipedia_summary` with `{'query': 'anillos de Saturno composición'}`




Ocurrió un error: Expecting value: line 1 column 1 (char 0)

Los anillos de **Saturno** están compuestos principalmente por **partículas de hielo de agua**, con trazas de roca y polvo. Estos fragmentos varían en tamaño, desde pequeñas partículas hasta bloques de varios metros de diámetro. Los anillos están divididos en varias secciones, como los anillos A, B y C, que tienen diferencias en densidad y composición.

> Finished chain.
Respuesta 2: Los anillos de **Saturno** están compuestos principalmente por **partículas de hielo de agua**, con trazas de roca y polvo. Estos fragmentos varían en tamaño, desde pequeñas partículas hasta bloques de varios metros de diámetro. Los anillos están divididos en varias secciones, como los anillos A, B y C, que tienen diferencias en densidad y composición.


¡Funcionó! El agente entendió que "sus anillos" se refería a los anillos de Saturno, porque la conversación anterior estaba en su contexto. El `verbose=True` nos muestra que el agente decidió buscar en Wikipedia "anillos de Saturno", combinando la nueva pregunta con el historial.

## Conclusiones

Añadir memoria a un agente de LangChain es sorprendentemente sencillo, pero increíblemente poderoso. Simplemente manteniendo una lista del historial de chat y pasándola en cada invocación, transformamos un agente de una sola respuesta en un verdadero **asistente conversacional**.

La clave es que los componentes de LangChain (`AgentExecutor`, el `ChatPromptTemplate` con su `MessagesPlaceholder`) ya están diseñados para buscar y utilizar la variable `chat_history` si se proporciona.

**Limitaciones:**
- **Gestión Manual**: En este ejemplo, actualizamos la lista `chat_history` manualmente. Para una aplicación real, querríamos encapsular esto en una clase o función.
- **Tamaño del Contexto**: Enviar el historial completo en cada llamada puede volverse costoso y exceder el límite de tokens del modelo en conversaciones muy largas.

En los próximos notebooks, exploraremos los **sistemas de memoria** que LangChain ofrece para gestionar estas limitaciones, como la memoria de búfer (`BufferMemory`) que automatiza la gestión del historial y la memoria de resumen (`SummaryMemory`) que condensa conversaciones largas para ahorrar tokens.